In [2]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import median_absolute_error
import pandas as pd
import numpy as np

In [3]:
dataset = pd.read_csv('train.csv')


X = dataset.iloc[:, :-1]
y = dataset.iloc[:, -1]

print(X.shape)
print(y.shape)

dataset.head(5)

(1460, 80)
(1460,)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [5]:
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming dataset, X, and y are already defined from previous cells

# Get feature names from the 'X' DataFrame (which contains features, excluding the target)
feature_names = X.columns.tolist()
n_features = len(feature_names)

# Determine grid size for subplots (e.g., 5 columns)
n_cols = 5
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 4))
axes = axes.flatten()

for i, feature_name in enumerate(feature_names):
    # Use 'y' (the target variable) for the y-axis and hue
    sns.scatterplot(x=dataset[feature_name], y=y, hue=y, ax=axes[i], alpha=0.6)
    axes[i].set_title(f'{feature_name} vs Meter', fontsize=10)
    axes[i].set_xlabel(feature_name, fontsize=8)
    axes[i].set_ylabel('Meter', fontsize=8)
    # The following lines for y-ticks are commented out because 'meter' is a continuous variable, not binary
    # axes[i].set_yticks([0, 1]) # Set y-ticks for binary target
    # axes[i].set_yticklabels(['Benign (0)', 'Malignant (1)']) # Label y-ticks

# Remove any unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.suptitle('Scatter Plots of Features vs Meter', y=1.02, fontsize=16)
plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [6]:
print(f"Data types of X:\n{X.dtypes}")
print(f"Are all columns in X numerical? {all(np.issubdtype(dtype, np.number) for dtype in X.dtypes)}")
print(f"Data type of y: {y.dtype}")
print(f"Is y fully numerical? {np.issubdtype(y.dtype, np.number)}")

Data types of X:
Id                 int64
MSSubClass         int64
MSZoning          object
LotFrontage      float64
LotArea            int64
                  ...   
MiscVal            int64
MoSold             int64
YrSold             int64
SaleType          object
SaleCondition     object
Length: 80, dtype: object
Are all columns in X numerical? False
Data type of y: int64
Is y fully numerical? True


In [8]:
X = pd.get_dummies(X, drop_first=True)

# Convert boolean columns to integer type (0 or 1)
for col in X.select_dtypes(include='bool').columns:
    X[col] = X[col].astype(int)

print(f"Data types of X after encoding and bool conversion:\n{X.dtypes}")
print(f"Are all columns in X numerical after encoding and bool conversion? {all(np.issubdtype(dtype, np.number) for dtype in X.dtypes)}")
print(f"Shape of X after encoding: {X.shape}")

Data types of X after encoding and bool conversion:
Id                         int64
MSSubClass                 int64
LotFrontage              float64
LotArea                    int64
OverallQual                int64
                          ...   
SaleCondition_AdjLand      int64
SaleCondition_Alloca       int64
SaleCondition_Family       int64
SaleCondition_Normal       int64
SaleCondition_Partial      int64
Length: 245, dtype: object
Are all columns in X numerical after encoding and bool conversion? True
Shape of X after encoding: (1460, 245)


In [11]:
from sklearn.model_selection import train_test_split

# First split: 67% for training, 33% for temp (validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(X, y,
                                                    test_size=0.33,
                                                    random_state=44,
                                                    shuffle=True)

# Second split: 50% of temp for validation, 50% for test (16.5% each of original)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp,
                                                test_size=0.5,
                                                random_state=44,
                                                shuffle=True)

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_val shape: {y_val.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (978, 245)
X_val shape: (241, 245)
X_test shape: (241, 245)
y_train shape: (978,)
y_val shape: (241,)
y_test shape: (241,)


### Handling Missing Values after Splitting Data

Now that the data has been split into training, validation, and test sets, it's crucial to handle any remaining missing values. I will use `SimpleImputer` with a `median` strategy. It's important to fit the imputer *only* on the training data (`X_train`) to prevent data leakage from the validation and test sets into the training process. Once fitted, the imputer will transform `X_train`, `X_val`, and `X_test`.

In [12]:
from sklearn.impute import SimpleImputer

# Create a median imputer
imputer = SimpleImputer(strategy='median')

# Fit the imputer on X_train only
imputer.fit(X_train)

# Transform X_train, X_val, and X_test
X_train_imputed = pd.DataFrame(imputer.transform(X_train), columns=X_train.columns, index=X_train.index)
X_val_imputed = pd.DataFrame(imputer.transform(X_val), columns=X_val.columns, index=X_val.index)
X_test_imputed = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns, index=X_test.index)

# Update the original variables with the imputed data
X_train = X_train_imputed
X_val = X_val_imputed
X_test = X_test_imputed

print("Missing values after imputation:")
print(f"X_train missing values: {X_train.isnull().sum().sum()}")
print(f"X_val missing values: {X_val.isnull().sum().sum()}")
print(f"X_test missing values: {X_test.isnull().sum().sum()}")

Missing values after imputation:
X_train missing values: 0
X_val missing values: 0
X_test missing values: 0


### Training and Evaluating the SVR Model

With the data now cleaned and split, I will proceed to train a Support Vector Regressor (SVR) model. After training on `X_train` and `y_train`, I'll evaluate the model's performance on the validation set (`X_val`, `y_val`) using Mean Absolute Error (MAE), Mean Squared Error (MSE), and Median Absolute Error (MedAE) to assess its accuracy.

In [22]:
# Initialize and train the SVR model
# Using a linear kernel as a starting point. Adjust C and epsilon as needed.
svr_model = SVR(kernel='linear', C=1.0, epsilon=0.2)
svr_model.fit(X_train, y_train)

# Make predictions on the validation set
y_pred_val = svr_model.predict(X_val)

# Evaluate the model on the validation set
mae_val = mean_absolute_error(y_val, y_pred_val)
mse_val = mean_squared_error(y_val, y_pred_val)
medae_val = median_absolute_error(y_val, y_pred_val)

print("SVR Model Performance on Validation Set:")
print(f"Mean Absolute Error (MAE): {mae_val:.2f}")
print(f"Mean Squared Error (MSE): {mse_val:.2f}")
print(f"Median Absolute Error (MedAE): {medae_val:.2f}")

SVR Model Performance on Validation Set:
Mean Absolute Error (MAE): 22768.12
Mean Squared Error (MSE): 1558881528.54
Median Absolute Error (MedAE): 14622.96


In [21]:
print("linear kernel")
print(f"Train R-squared score: {svr_model.score(X_train, y_train):.4f}")
print(f"Validation R-squared score: {svr_model.score(X_val, y_val):.4f}")
print(f"Test R-squared score: {svr_model.score(X_test, y_test):.4f}")

linear kernel
Train R-squared score: -0.0541
Validation R-squared score: -0.0230
Test R-squared score: -0.0704
